## Importações

In [20]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import seaborn as sns
import matplotlib.pyplot as plt

## Carregar dataset

In [21]:
url = "https://raw.githubusercontent.com/latcj/fiap-data-analytics-tc4-obesity-prediction/refs/heads/main/data/Obesity.csv"
df = pd.read_csv(url)

## Arredonda valores inteiros

In [22]:
columns_to_int = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
for col in columns_to_int:
    df[col] = df[col].round().astype(int)

## Remocao da altura e peso

In [23]:
# df['BMI'] = df['Weight'] / (df['Height']**2)
df = df.drop(columns=["Height", "Weight"])

## Separação treino & teste

In [24]:
X = df.drop('Obesity', axis=1)
y = df['Obesity']

## Identificação das Colunas

In [25]:
categoricas = X.select_dtypes(
    include='object'
).columns.tolist()

numericas = X.select_dtypes(
    exclude='object'
).columns.tolist()

print(categoricas)
print(numericas)

['Gender', 'family_history', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


## Pré-processamento

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numericas
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            categoricas
        )
    ]
)
y = LabelEncoder().fit_transform(y)

## Split

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## Modelos

In [28]:
modelos = {
    "Logistic Regression":
        LogisticRegression(max_iter=5000),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric='mlogloss'
        )
}

## Comparação dos modelos

In [29]:
resultados = {}
pipelines = {}

for nome, modelo in modelos.items():

    pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', modelo)
    ])

    pipeline.fit(
        X_train,
        y_train
    )

    pred = pipeline.predict(X_test)
    acc = accuracy_score(
        y_test,
        pred
    )

    resultados[nome] = acc
    pipelines[nome] = pipeline

    print(nome, round(acc,4))

Logistic Regression 0.6052
Random Forest 0.7967
XGBoost 0.7943


## Melhor Modelo

In [30]:
resultado_df = pd.DataFrame(
    resultados.items(),
    columns=['Modelo','Accuracy']
)

resultado_df.sort_values(
    by='Accuracy',
    ascending=False,
    inplace=True
)

resultado_df

,Modelo,Accuracy
1,Random Forest,0.796690
2,XGBoost,0.794326
0,Logistic Regression,0.605201


## Avaliação final

In [31]:
melhor_modelo = resultado_df.iloc[0]['Modelo']
best_pipeline = pipelines[melhor_modelo]
pred = best_pipeline.predict(X_test)

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.83      0.93      0.88        54
           1       0.60      0.66      0.63        58
           2       0.78      0.81      0.80        70
           3       0.91      0.82      0.86        60
           4       0.98      0.98      0.98        65
           5       0.73      0.69      0.71        58
           6       0.74      0.67      0.70        58

    accuracy                           0.80       423
   macro avg       0.80      0.79      0.79       423
weighted avg       0.80      0.80      0.80       423



## Exportar modelo

In [33]:
joblib.dump(
    best_pipeline,
    "obesity_model.pkl"
)

['obesity_model.pkl']

In [35]:
from google.colab import files

files.download('obesity_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>